In [1]:
#Cell 1 — load selected-feature data:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import joblib

from src.utils import load_config, set_seed
from src.models.advanced import tune_random_forest
from src.evaluation.metrics import evaluate_model, print_metrics

cfg = load_config()
set_seed(cfg["project"]["random_state"])
processed_dir = cfg["paths"]["processed_dir"]
models_dir = cfg["paths"]["models_dir"]

X_train_selected = pd.read_parquet(os.path.join(processed_dir, "X_train_selected.parquet"))
X_test_selected = pd.read_parquet(os.path.join(processed_dir, "X_test_selected.parquet"))
y_train = pd.read_parquet(os.path.join(processed_dir, "y_train.parquet"))["label_binary"]
y_test = pd.read_parquet(os.path.join(processed_dir, "y_test.parquet"))["label_binary"]

print("Selected feature data loaded:", X_train_selected.shape)

Selected feature data loaded: (178464, 20)


In [2]:
#Cell 2 — run the light tuning (may take a few minutes — this is the heaviest step so far):
seed = cfg["project"]["random_state"]

best_rf, best_params, tune_time = tune_random_forest(X_train_selected, y_train, random_state=seed)

print(f"Tuning completed in {tune_time:.1f} seconds")
print("\nBest hyperparameters found:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Tuning completed in 482.4 seconds

Best hyperparameters found:
  n_estimators: 150
  min_samples_split: 5
  min_samples_leaf: 1
  max_features: sqrt
  max_depth: 15


In [3]:
#Cell 3 — evaluate the tuned model on test set:
final_metrics = evaluate_model(best_rf, X_test_selected, y_test, model_name="Final RF (tuned, 20 features)")
final_metrics["train_time_sec"] = round(tune_time, 4)

print_metrics(final_metrics)

RESULTS: Final RF (tuned, 20 features)
Accuracy   : 0.9999
Precision  : 0.9999
Recall     : 0.9999
F1-score   : 0.9999
ROC-AUC    : 1.0000
PR-AUC     : 1.0000
FPR        : 0.0001
FNR        : 0.0001
------------------------------------------------------------
Confusion Matrix:
              Predicted BENIGN   Predicted DDoS
Actual BENIGN        19,011                 2
Actual DDoS               2            25,601
------------------------------------------------------------
Training time    : 482.3671 sec
Prediction time  : 0.3011 sec


In [4]:
#Cell 4 — save the final model:
import json

joblib.dump(best_rf, os.path.join(models_dir, "final_model_random_forest.joblib"))

metrics_dir = cfg["paths"]["metrics_dir"]
pd.DataFrame([final_metrics]).to_csv(os.path.join(metrics_dir, "final_model_metrics.csv"), index=False)

reports_dir = cfg["paths"]["reports_dir"]
with open(os.path.join(reports_dir, "final_model_hyperparameters.json"), "w") as f:
    json.dump(best_params, f, indent=2)

print("Final model saved.")
print("This model will be used for Step 8 (evaluation plots), Step 9 (SHAP/LIME),")
print("and Step 10/11 (cross-dataset testing).")

Final model saved.
This model will be used for Step 8 (evaluation plots), Step 9 (SHAP/LIME),
and Step 10/11 (cross-dataset testing).
